In [1]:
# Lecture 4 — Scatter & Bubble Charts: Gapminder
# Source: Gapminder Foundation, bundled with Plotly Express.
import math
import pandas as pd
import plotly.express as px
from IPython.display import display, Markdown

df = px.data.gapminder()
print(f'Loaded: {len(df)} rows; {df.country.nunique()} countries')
display(df.head())
df.info()
print('Years:', sorted(df.year.unique()))
print('Continents:', sorted(df.continent.unique()))
display(df.describe().round(1))

display(Markdown('''## Design choices from Lecture 4
Colour and direction guide attention; grey provides context. Short legends and selective labels reduce memory load. GDP uses a log scale so equal ratios have equal spacing. Bubble **area** represents population, with a capped size and transparency. Titles state data-supported findings. These descriptive comparisons do not establish causation.

The bundled historical data end in 2007; these charts do not describe present-day conditions.'''))

Loaded: 1704 rows; 142 countries


,country,continent,year,lifeExp,pop,gdpPercap,iso_alpha,iso_num
0,Afghanistan,Asia,1952,28.801,8425333,779.445314,AFG,4
1,Afghanistan,Asia,1957,30.332,9240934,820.853030,AFG,4
2,Afghanistan,Asia,1962,31.997,10267083,853.100710,AFG,4
3,Afghanistan,Asia,1967,34.020,11537966,836.197138,AFG,4
4,Afghanistan,Asia,1972,36.088,13079460,739.981106,AFG,4


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   object 
 1   continent  1704 non-null   object 
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   object 
 7   iso_num    1704 non-null   int64  
dtypes: float64(2), int64(3), object(3)
memory usage: 106.6+ KB
Years: [np.int64(1952), np.int64(1957), np.int64(1962), np.int64(1967), np.int64(1972), np.int64(1977), np.int64(1982), np.int64(1987), np.int64(1992), np.int64(1997), np.int64(2002), np.int64(2007)]
Continents: ['Africa', 'Americas', 'Asia', 'Europe', 'Oceania']


,year,lifeExp,pop,gdpPercap,iso_num
count,1704.0,1704.0,1.704000e+03,1704.0,1704.0
mean,1979.5,59.5,2.960121e+07,7215.3,425.9
std,17.3,12.9,1.061579e+08,9857.5,248.3
min,1952.0,23.6,6.001100e+04,241.2,4.0
25%,1965.8,48.2,2.793664e+06,1202.1,208.0
50%,1979.5,60.7,7.023596e+06,3531.8,410.0
75%,1993.2,70.8,1.958522e+07,9325.5,638.0
max,2007.0,82.6,1.318683e+09,113523.1,894.0


## Design choices from Lecture 4
Colour and direction guide attention; grey provides context. Short legends and selective labels reduce memory load. GDP uses a log scale so equal ratios have equal spacing. Bubble **area** represents population, with a capped size and transparency. Titles state data-supported findings. These descriptive comparisons do not establish causation.

The bundled historical data end in 2007; these charts do not describe present-day conditions.

In [2]:
# Task 1 — Europe, 2002 and 2007: colour encodes year only
comparison = df.loc[(df.continent == 'Europe') & df.year.isin([2002, 2007])].copy()
comparison['Year'] = comparison.year.astype(str)
wide = comparison.pivot(index='country', columns='year', values=['gdpPercap', 'lifeExp'])
gdp_change = (wide['gdpPercap'][2007] / wide['gdpPercap'][2002] - 1) * 100
life_change = wide['lifeExp'][2007] - wide['lifeExp'][2002]
both_up = int(((gdp_change > 0) & (life_change > 0)).sum())

display(Markdown('## Task 1 — European countries over time'))
fig1 = px.scatter(comparison, x='gdpPercap', y='lifeExp', color='Year',
    color_discrete_map={'2002': '#D55E00', '2007': '#0072B2'},
    category_orders={'Year': ['2002', '2007']}, hover_name='country',
    hover_data={'Year': True, 'gdpPercap': ':,.0f', 'lifeExp': ':.1f'},
    log_x=True, template='plotly_white',
    labels={'gdpPercap': 'GDP per capita (USD, log scale)', 'lifeExp': 'Life expectancy (years)'},
    title=f'{both_up} of {len(wide)} European countries gained<br>in both wealth and longevity (2002–2007)',
    height=650)
fig1.update_traces(marker=dict(size=9, opacity=0.85, line=dict(width=0.5, color='white')))
# On Plotly log axes, annotation positions use log10 coordinates.
# Each subtle arrow connects the same country from 2002 to 2007.
for country, row in wide.iterrows():
    fig1.add_annotation(x=math.log10(row[('gdpPercap', 2007)]), y=row[('lifeExp', 2007)],
        ax=math.log10(row[('gdpPercap', 2002)]), ay=row[('lifeExp', 2002)],
        xref='x', yref='y', axref='x', ayref='y', text='', showarrow=True,
        arrowhead=2, arrowsize=0.8, arrowwidth=1, arrowcolor='#A0A0A0', opacity=0.6)
for country, offset in [('Turkey', 35), ('Albania', -35)]:
    row = comparison.loc[(comparison.country == country) & (comparison.year == 2007)].iloc[0]
    fig1.add_annotation(x=math.log10(row.gdpPercap), y=row.lifeExp,
        text=country + ' (2007)', showarrow=True, ax=45, ay=offset,
        arrowcolor='#0072B2', font=dict(color='#0072B2'), bgcolor='white')
fig1.update_layout(font=dict(family='Arial', size=13),
    legend=dict(orientation='h', y=1.08, x=0, title='Year'),
    margin=dict(t=120, b=80))
fig1.update_xaxes(showgrid=True, gridcolor='#EEEEEE')
fig1.update_yaxes(showgrid=False)
fig1.add_annotation(x=0, y=-0.15, xref='paper', yref='paper', showarrow=False,
    text='Arrows: 2002 → 2007 for each country. Source: Gapminder / Plotly.', xanchor='left')
fig1.show()
print(f'Median country GDP gain: {gdp_change.median():.1f}%')
print(f'Median country life expectancy gain: {life_change.median():.2f} years')
display(pd.DataFrame({'GDP change (%)': gdp_change, 'Life expectancy change (years)': life_change}).round(2))
display(Markdown('**Interpretation:** The paired arrows show within-country movement, while the two colours distinguish years. Medians give every country equal weight. The pattern does not prove that higher GDP caused the improvement in life expectancy. Turkey follows the dataset’s Europe classification.'))


## Task 1 — European countries over time

Median country GDP gain: 18.6%
Median country life expectancy gain: 0.94 years


,GDP change (%),Life expectancy change (years)
country,,
Albania,28.95,0.77
Austria,11.44,0.85
Belgium,10.52,1.12
Bosnia and Herzegovina,23.71,0.76
Bulgaria,38.77,0.86
Croatia,25.72,0.87
Czech Republic,29.76,0.98
Denmark,9.67,1.15
Finland,17.74,0.94


**Interpretation:** The paired arrows show within-country movement, while the two colours distinguish years. Medians give every country equal weight. The pattern does not prove that higher GDP caused the improvement in life expectancy. Turkey follows the dataset’s Europe classification.

In [3]:
# Task 2 — All 142 countries in 2007, highlighting Europe
world = df.loc[df.year == 2007].copy()
world['Group'] = world.continent.where(world.continent.eq('Europe'), 'Other continents')
world = world.sort_values('pop', ascending=False)
europe = world.loc[world.continent == 'Europe']
world_median = world.lifeExp.median()
europe_median = europe.lifeExp.median()
above = int((europe.lifeExp > world_median).sum())
display(Markdown('## Task 2 — Europe in the global context (2007)'))
fig2 = px.scatter(world, x='gdpPercap', y='lifeExp', size='pop', color='Group',
    category_orders={'Group': ['Other continents', 'Europe']},
    color_discrete_map={'Other continents': '#B8B8B8', 'Europe': '#0072B2'},
    size_max=60, log_x=True, hover_name='country', custom_data=['continent', 'pop'],
    labels={'gdpPercap': 'GDP per capita (USD, log scale)', 'lifeExp': 'Life expectancy (years)', 'pop': 'Population'},
    template='plotly_white', height=550,
    title=f'Europe: {above}/{len(europe)} countries above<br>the global median life expectancy (2007)')
fig2.update_traces(marker=dict(sizemode='area', line=dict(width=0.5, color='white')),
    hovertemplate='<b>%{hovertext}</b><br>Continent: %{customdata[0]}<br>GDP per capita: $%{x:,.0f}<br>Life expectancy: %{y:.1f} years<br>Population: %{customdata[1]:,.0f}<extra></extra>')
fig2.update_traces(selector=dict(name='Other continents'), marker_opacity=0.45)
fig2.update_traces(selector=dict(name='Europe'), marker_opacity=0.85)
fig2.add_hline(y=world_median, line_dash='dot', line_color='#777777', line_width=1,
    annotation_text=f'Global country median: {world_median:.1f} years',
    annotation_position='bottom left')
fig2.add_annotation(x=math.log10(18000), y=57, showarrow=False,
    text=f'<b>Europe</b><br>Median life expectancy: {europe_median:.1f} years<br>{above}/{len(europe)} countries above the global median',
    font=dict(color='#0072B2', size=13), bgcolor='white', bordercolor='#0072B2', borderpad=8)
fig2.update_layout(font=dict(family='Arial', size=13),
    legend=dict(orientation='h', y=1.08, x=0, title=''), margin=dict(t=120, b=90))
fig2.update_xaxes(showgrid=True, gridcolor='#EEEEEE')
fig2.update_yaxes(showgrid=False)
fig2.add_annotation(x=0, y=-0.16, xref='paper', yref='paper', showarrow=False, xanchor='left',
    text='Bubble area = population; blue = Europe. Medians weight countries equally.<br>Source: Gapminder / Plotly.')
fig2.show()
print(f'Europe median: {europe_median:.2f} years; global country median: {world_median:.2f} years')
display(Markdown('**Interpretation:** Europe clusters toward higher income and longer life expectancy, while the grey bubbles retain the global context. Bubble area encodes population; it does not weight the median. Regional differences remain descriptive and do not demonstrate a causal effect of income.'))

# Validate the actual plotted data and encodings.
assert len(comparison) == 60 and comparison.country.nunique() == 30
assert set(comparison.year) == {2002, 2007}
assert len(world) == world.country.nunique() == 142
assert sum(len(t.x) for t in fig1.data) == 60
assert sum(len(t.x) for t in fig2.data) == 142
assert fig1.layout.xaxis.type == fig2.layout.xaxis.type == 'log'
assert all(t.marker.sizemode == 'area' for t in fig2.data)
assert world[['gdpPercap', 'lifeExp', 'pop']].notna().all().all()
fig1.write_html('/kaggle/working/task1_europe_change.html', include_plotlyjs=True)
fig2.write_html('/kaggle/working/task2_europe_bubbles.html', include_plotlyjs=True)


## Task 2 — Europe in the global context (2007)

Europe median: 78.61 years; global country median: 71.94 years


**Interpretation:** Europe clusters toward higher income and longer life expectancy, while the grey bubbles retain the global context. Bubble area encodes population; it does not weight the median. Regional differences remain descriptive and do not demonstrate a causal effect of income.